# Customer Intelligence & Support Copilot
## End-to-End Capstone Analysis Notebook

**Scenario:** Synthetic grocery quick-commerce data for Swiggy Instamart. This notebook inspects data ingestion, cleaning rules, storage in SQLite, customer segmentation, support ticket triage, policy RAG, and automated refund tools.

In [2]:
from pathlib import Path
import sys
import pandas as pd
import sqlite3

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: c:\Users\Admin\Desktop\instamart-copilot
Root files: ['.gitignore', 'Instamart_Copilot_Analysis.ipynb', 'generate_data.py', 'mock_llm.py', 'part_a_pipeline.py', 'part_b_pipeline.py', 'part_c_pipeline.py', 'part_d_pipeline.py', 'requirements.txt', 'run_project.py', 'test_queries.py']


## Part A — Data Foundation & SQLite Storage
Inspecting the cleaned dataset and SQLite database tables (`customers`, `products`, `orders`).

In [3]:
db_path = ROOT / "data" / "instamart.db"

with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    
    # Run a sample SQL query joining orders, customers, and products
    q_revenue = pd.read_sql_query("""
        SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)

display(q_revenue.head(10))
print("Total Revenue (INR):", round(q_revenue["revenue"].sum(), 2))

,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


Total Revenue (INR): 2998668.8


## Part B — Customer Segments and Lapse Prediction
Loading generated customer segments and inspecting the Random Forest lapse model outputs.

In [4]:
segments_df = pd.read_csv(ROOT / "outputs" / "customer_segments.csv")
retention_df = pd.read_csv(ROOT / "outputs" / "retention_list.csv")

print("Customer Segments Head:")
display(segments_df.head())

print("\nTop Customers at Risk of Lapse (Retention List):")
display(retention_df.head())

Customer Segments Head:


,customer_id,city,segment,pca_1,pca_2
0,C0001,Chennai,Rare,-0.764756,-0.686357
1,C0002,Delhi,Premium,0.820685,1.170604
2,C0003,Bengaluru,Frequent,1.284173,-1.156579
3,C0004,Bengaluru,Rare,-0.623913,-0.898790
4,C0005,Bengaluru,Rare,-2.460958,0.391996



Top Customers at Risk of Lapse (Retention List):


,customer_id,city,frequency,monetary,lapse_probability
0,C0057,Pune,1.0,285.0,0.77
1,C0310,Delhi,1.0,329.4,0.74
2,C0156,Mumbai,1.0,280.0,0.73
3,C0295,Delhi,2.0,100.2,0.59
4,C1038,Delhi,2.0,441.0,0.57


## Part C — Support Ticket Triage with Privacy Guardrails
Inspecting PII-masked support tickets and mock LLM classification results.

In [5]:
triage_df = pd.read_csv(ROOT / "outputs" / "triage_results.csv")
print(f"Total Triaged Tickets: {len(triage_df)}")
display(triage_df.head(10))

print("\nPredicted Category Counts:")
display(triage_df["predicted_category"].value_counts())

Total Triaged Tickets: 164


,ticket_id,customer_id,true_category,predicted_category,reason
0,T0024,C0486,fraud,fraud,keyword match
1,T0073,C0080,billing,billing,keyword match
2,T0095,C1087,technical,billing,keyword match
3,T0159,C1172,none,technical,keyword match
4,T0080,C0910,fraud,billing,keyword match
5,T0124,C0139,fraud,billing,keyword match
6,T0061,C0812,billing,billing,keyword match
7,T0118,C0355,delivery,delivery,keyword match
8,T0037,C1156,billing,billing,keyword match
9,T0002,C0881,delivery,delivery,keyword match



Predicted Category Counts:


predicted_category
billing      60
delivery     42
technical    30
fraud        23
shipping      9
Name: count, dtype: int64

## Part D — Policy RAG Assistant and Refund Ledger
Inspecting automated agent trace logs and the refund ledger.

In [ ]:
import json

ledger_path = ROOT / "outputs" / "refund_ledger.jsonl"
if ledger_path.exists():
    ledger_entries = []
    with open(ledger_path, "r", encoding="utf-8") as f:
        for line in f:
            ledger_entries.append(json.loads(line))
    ledger_df = pd.DataFrame(ledger_entries)
    print("Refund Ledger Entries:")
    display(ledger_df)
else:
    print("Refund ledger not found yet. Run part_d_pipeline.py first.")

Refund Ledger Entries:


,scenario_id,customer_id,refund_inr
0,S1,C0007,128.25
1,S2,C0008,135.00


: 